# Evaluate uploaded v2 MVTec perturbations on AnomalyCLIP

This notebook evaluates the fixed focal-plus-Dice MVTec bundle without regenerating attacks. It is separate from the original and full attack notebooks, so both existing workflows remain unchanged. Attach the uploaded perturbation dataset and MVTec AD, then enable a GPU and Internet.

In [ ]:
# Reproducible code/model setup
import subprocess, sys
from pathlib import Path
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
EXPERIMENT_REPO = 'https://github.com/Parsagh05/adversarial-robustness.git'
EXPERIMENT_REF = 'main'  # Prefer the exact commit used for the final comparison.
ANOMALYCLIP_COMMIT = '3911738c0867544f545a076ad78f3f11d9ecbfdf'
def checkout(url, destination, revision):
    if not (destination / '.git').is_dir():
        subprocess.run(['git', 'clone', '--filter=blob:none', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', '--force', 'FETCH_HEAD'], check=True)
checkout(EXPERIMENT_REPO, EXPERIMENT_ROOT, EXPERIMENT_REF)
checkout('https://github.com/zqhang/AnomalyCLIP.git', ANOMALYCLIP_ROOT, ANOMALYCLIP_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'requirements.txt')], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print('Evaluation commit:', subprocess.check_output(['git', '-C', str(EXPERIMENT_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Find and validate exactly one uploaded v2 MVTec bundle
import pandas as pd
from blackbox_evaluation_pipeline.universal_eval.artifacts import load_manifest
EXPLICIT_BUNDLE_ROOT = None  # Example: Path('/kaggle/input/my-v2-dataset/canonical_clip_per_dataset_mvtec_segmentation_loss_v2')
def is_v2_mvtec_bundle(path):
    manifest = path / 'attack_manifest.csv'
    diagnostics = path / 'optimization_diagnostics.csv'
    if not (manifest.is_file() and diagnostics.is_file() and (path / 'evaluation_test_indices.csv').is_file()):
        return False
    try:
        frame = pd.read_csv(manifest)
        return len(frame) == 6 and set(frame['scope']) == {'dataset'} and set(frame['source_dataset']) == {'mvtec'} and set(frame['target_dataset']) == {'mvtec'} and set(frame['local_objective']) == {'target_class_focal_plus_soft_dice'}
    except Exception:
        return False
if EXPLICIT_BUNDLE_ROOT is not None:
    candidates = [Path(EXPLICIT_BUNDLE_ROOT)]
else:
    candidates = sorted({path.parent for path in Path('/kaggle/input').rglob('attack_manifest.csv') if is_v2_mvtec_bundle(path.parent)})
valid = [path for path in candidates if is_v2_mvtec_bundle(path)]
if len(valid) != 1:
    raise RuntimeError(f'Expected exactly one uploaded v2 MVTec bundle, found: {valid}')
ARTIFACTS_ROOT = valid[0]
artifacts = load_manifest(ARTIFACTS_ROOT, scopes=('per_dataset',), sources=('mvtec',), targets=('mvtec',), verify_files=True)
if len(artifacts) != 6:
    raise RuntimeError(f'Expected six attack conditions, loaded {len(artifacts)}')
diagnostics = pd.read_csv(ARTIFACTS_ROOT / 'optimization_diagnostics.csv')
passed = diagnostics['convergence_check_passed'].astype(str).str.lower().eq('true')
if not passed.all():
    raise RuntimeError('The uploaded bundle contains a failed convergence condition.')
display(diagnostics[['direction', 'loss_mode', 'initial_total_loss', 'final_total_loss', 'total_loss_reduction']])
print('Validated bundle:', ARTIFACTS_ROOT)

In [ ]:
# Resolve MVTec and the AnomalyCLIP target checkpoint
import torch
def find_mvtec():
    preferred = [Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'), Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')]
    for path in [*preferred, *Path('/kaggle/input').rglob('mvtec_anomaly_detection')]:
        if (path / 'bottle' / 'test').is_dir():
            return path.resolve()
    raise FileNotFoundError('Attach the MVTec AD Kaggle dataset.')
MVTEC_ROOT = find_mvtec()
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f'AnomalyCLIP checkpoint missing: {CHECKPOINT}')
print('GPU:', torch.cuda.get_device_name(0))
print('MVTec:', MVTEC_ROOT)
print('Checkpoint:', CHECKPOINT)

In [ ]:
# Evaluate all six new conditions. This does not optimize or alter the perturbations.
import json
from blackbox_evaluation_pipeline import EvaluationConfig, run_evaluation
def normalized(value):
    return ''.join(c for c in value.lower() if c.isalnum())
threshold = EXPERIMENT_ROOT / 'blackbox_evaluation_pipeline' / 'thresholds' / 'anomalyclip' / 'mvtec' / 'category_thresholds.json'
if not threshold.is_file():
    raise FileNotFoundError(f'Committed AnomalyCLIP/MVTec threshold artifact missing: {threshold}')
OUTPUT_ROOT = WORKING / 'anomalyclip_v2_mvtec_results'
SAMPLES_ROOT = WORKING / 'anomalyclip_v2_mvtec_samples'
config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT), mvtec_root=str(MVTEC_ROOT), visa_root=None,
    output_root=str(OUTPUT_ROOT), model_name='anomalyclip',
    model_kwargs_by_target={'mvtec': {'repository_root': str(ANOMALYCLIP_ROOT), 'checkpoint_path': str(CHECKPOINT), 'clip_download_root': str(WORKING / 'clip_cache')}},
    thresholds_by_target={'mvtec': str(threshold)}, device='cuda', batch_size=2,
    metric_size=518, anomaly_map_sigma=4.0, aupro_fpr_limit=0.30, aupro_max_thresholds=200,
    verify_checksums=True, save_predictions=True, save_qualitative_samples=True, qualitative_output_root=str(SAMPLES_ROOT),
    attack_scopes=('per_dataset',), source_datasets=('mvtec',), target_datasets=('mvtec',),
    attack_categories=None, attack_directions=None, attack_loss_modes=None, max_conditions=None,
    run_notes='Uploaded focal-plus-Dice v2 MVTec perturbations; fixed evaluation IDs.'
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)

In [ ]:
# Preview and package results
import shutil
summary = pd.read_csv(SUMMARY_PATH)
columns = ['source_dataset', 'target_dataset', 'direction', 'loss_mode', 'category', 'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc', 'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc', 'clean_aupro', 'adversarial_aupro', 'delta_aupro', 'targeted_attack_success_rate']
display(summary[columns])
results_zip = shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name)
samples_zip = shutil.make_archive(str(SAMPLES_ROOT), 'zip', root_dir=SAMPLES_ROOT.parent, base_dir=SAMPLES_ROOT.name)
print('Numerical results:', results_zip)
print('Qualitative samples:', samples_zip)